In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objs as go
import gzip
import h5py
import scanpy as sc
import scipy
import mira
import torch
import anndata as ad

# merge samples

In [ ]:
filenames = glob.glob("/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/per_sample_outs/*/count/sample_filtered_feature_bc_matrix.h5")

adatas=[]
for filename in filenames:
    adatas.append(sc.read_10x_h5(filename))
    adatas[-1].var_names_make_unique()
    adatas[-1].obs['group']=filename.split('/')[-3]

In [ ]:
adata = adatas[0].concatenate(adatas[1:],index_unique=None)

In [ ]:
def remove_var_columns(adata: ad.AnnData, columns_to_remove: list[str]):
  """
  Removes specified columns from the .var DataFrame of an AnnData object.

  Args:
      adata: The AnnData object.
      columns_to_remove: A list of column names to remove.
  """

  var_columns = adata.var.columns
  columns_to_keep = [col for col in var_columns if col not in columns_to_remove]
  adata.var = adata.var[columns_to_keep]


# Example usage:
columns_to_remove = ['pattern', 'read', 'sequence']  # List of columns to remove
remove_var_columns(adata, columns_to_remove)
adata.var

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=20)

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4,
             multi_panel=True
            )

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 10000, :]
adata = adata[adata.obs.pct_counts_mt < 15, :]

In [ ]:
adata.obs_names_make_unique()

In [ ]:
rawdata = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.layers['counts'] = rawdata

In [ ]:
sc.pp.highly_variable_genes(adata, min_disp = 0.2)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
adata

In [ ]:
adata.var['highly_variable'].value_counts()

In [ ]:
adata.write("/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged.h5ad")